# 02 — Exploratory Data Analysis
Run **after** `01_Feature_Engineering.ipynb`.

**Input:** `forecasting_features.parquet`

Covers: data overview, operating-calendar check, demand distribution & sparsity,
ABC breakdown, demand concentration (Pareto), and feature correlations.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

panel = pd.read_parquet('forecasting_features.parquet')
print(panel.shape)
panel.head()

In [ ]:
import os as _os
# ── Image output folder ─────────────────────────────────────────────────────
# All figures will be saved to a subfolder called "figures" inside the
# directory where you run this notebook.  Change FIGURES_DIR if you prefer
# a different path.
FIGURES_DIR = _os.path.join(_os.getcwd(), "figures")
_os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"[INFO] Figures will be saved to: {FIGURES_DIR}")


## 1. Data overview

In [ ]:
print("Rows:", len(panel))
print("Unique (Reference, Size) combos:", panel.groupby(['Reference','Size (US)']).ngroups)
print("Unique References:", panel['Reference'].nunique())
print("Business days:", panel['date'].nunique(), "(", panel['date'].min().date(), "->", panel['date'].max().date(), ")")
print("\nTarget ('picks') summary:")
panel['picks'].describe()

## 2. Operating calendar — confirms the warehouse is a weekday operation

In [ ]:
dow_counts = panel.groupby(panel['date'].dt.dayofweek)['picks'].sum()
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
plt.figure(figsize=(7,4))
sns.barplot(x=[dow_labels[i] for i in dow_counts.index], y=dow_counts.values, color='#4C72B0')
plt.title('Total Picks by Day of Week')
plt.ylabel('Total picks'); plt.xlabel('')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_picks_by_dow.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Daily total demand over time

In [ ]:
daily_total = panel.groupby('date')['picks'].sum()
plt.figure(figsize=(12,4))
plt.plot(daily_total.index, daily_total.values, color='#55A868', linewidth=1)
plt.title('Total Picks per Business Day (all products)')
plt.ylabel('Total picks'); plt.xlabel('Date')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_daily_total_demand.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Target distribution & sparsity
The core modeling challenge: this is zero-inflated intermittent demand.

In [ ]:
zero_rate = (panel['picks']==0).mean()
print(f"Zero-rate: {zero_rate:.1%}")

fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.histplot(panel.loc[panel['picks']>0,'picks'], bins=40, ax=axes[0], color='#C44E52')
axes[0].set_title('Distribution of Picks (non-zero days only)')
axes[0].set_xlabel('Picks'); axes[0].set_yscale('log')

axes[1].pie([zero_rate, 1-zero_rate], labels=['Zero-pick days','Active days'],
            autopct='%1.1f%%', colors=['#DD8452','#4C72B0'])
axes[1].set_title('Share of Zero vs Active (Reference,Size,Day) Cells')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_demand_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. ABC classification breakdown
Note: `Sector` was checked and found to be **constant across all 208 products** (every row is
`"PF"`) — it carries zero predictive information, so it's excluded from the EDA and from the
feature set used for modeling. Only `ABCCOD` is a real categorical attribute here.

In [ ]:
abc_picks = panel.groupby('ABCCOD', observed=True)['picks'].sum().sort_values(ascending=False)
plt.figure(figsize=(6,4))
sns.barplot(x=abc_picks.index.astype(str), y=abc_picks.values, color='#4C72B0')
plt.title('Total Picks by ABC Class'); plt.ylabel('Total picks')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_abc_breakdown.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Demand concentration (Pareto curve)
Checks the "few products drive most picks" pattern that motivates slotting optimization.

In [ ]:
prod_totals = panel.groupby('Reference')['picks'].sum().sort_values(ascending=False)
cum_share = prod_totals.cumsum() / prod_totals.sum()
pct_products = np.arange(1, len(prod_totals)+1) / len(prod_totals)

n80 = (cum_share <= 0.80).sum()
print(f"{n80} of {len(prod_totals)} products ({n80/len(prod_totals):.1%}) drive 80% of all picks")

plt.figure(figsize=(7,5))
plt.plot(pct_products*100, cum_share*100, color='#4C72B0', linewidth=2)
plt.axhline(80, color='gray', linestyle='--', linewidth=1)
plt.axvline(n80/len(prod_totals)*100, color='gray', linestyle='--', linewidth=1)
plt.xlabel('% of Products (ranked by demand)'); plt.ylabel('Cumulative % of Picks')
plt.title('Demand Concentration Across Products')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_pareto_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Feature correlations with target

In [ ]:
feat_cols = ['lag_1','lag_5','lag_10','rmean_5','rmean_20','freq_10','freq_20',
             'nz_mean_20','expanding_mean','days_since_last','series_age']
corr = panel[feat_cols + ['picks']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink':.8})
plt.title('Correlation Matrix: Key Features vs Picks')
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_feature_correlation.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Size (US) distribution of demand

In [ ]:
size_picks = panel.groupby('Size (US)')['picks'].sum().sort_index()
plt.figure(figsize=(10,4))
sns.barplot(x=size_picks.index.astype(str), y=size_picks.values, color='#8172B2')
plt.title('Total Picks by Size (US)'); plt.ylabel('Total picks'); plt.xlabel('Size (US)')
plt.xticks(rotation=90)
plt.tight_layout(); plt.savefig(_os.path.join(FIGURES_DIR, "02_picks_by_size.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Extra graphic: Summary statistics heatmap ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd, numpy as np, os as _os

stats_df = panel.groupby('ABCCOD', observed=True)['picks'].agg(
    ['mean','std','median', lambda x: (x==0).mean()]).reset_index()
stats_df.columns = ['ABC Class','Mean Daily Picks','Std Dev','Median','Zero Rate']

fig, ax = plt.subplots(figsize=(8, 2 + len(stats_df)*0.6))
ax.axis('off')
tbl = ax.table(
    cellText=stats_df.round(3).values,
    colLabels=stats_df.columns,
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2C4770')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#EEF2F8')
plt.title('Demand Statistics by ABC Class', pad=12, fontweight='bold')
plt.tight_layout()
plt.savefig(_os.path.join(FIGURES_DIR, "02_abc_stats_table.png"), dpi=150, bbox_inches="tight")
plt.show()
